# PANDAS I — COMPLEMENTO CODEX

## Objetivos de aprendizaje

#### Nota Codex — Correspondencia con el curso

Acompaña [Pandas I: estructuras de datos](01_Pandas_Base_I_EstructurasDeDatos.ipynb). Estudia las secciones 1 y 2 después de «De DataFrame a Series»; las secciones 3 y 4 después de crear Series y DataFrames. La sección 5 precisa «Tipos de datos». Puedes ejecutar este cuaderno completo sin archivos externos.

Al terminar podrás:

1. Anticipar si una operación devuelve un escalar, una Series o un DataFrame.
2. Distinguir tamaño, dimensiones y etiquetas.
3. Explicar por qué Pandas alinea datos por índice.
4. Crear estructuras con tipos explícitos y faltantes sin confundirlos con cero.

## Preparación

In [1]:
import pandas as pd
from IPython.display import display
print("Versión de Pandas:", pd.__version__)

Versión de Pandas: 2.3.1


## 1. Una columna puede ser Series o DataFrame

#### Nota Codex — Concepto y uso

Trabajaremos con tres productos. El diccionario aporta columnas y las listas aportan sus valores, en el mismo orden. El índice guarda códigos: no son cantidades ni posiciones.

Un **escalar** es un solo valor; una **Series** tiene un eje; un **DataFrame** tiene dos. Tener una sola columna no vuelve automáticamente un DataFrame una Series. Elige la estructura según lo que la operación siguiente necesite.

In [2]:
productos = pd.DataFrame({
    'producto': ['Cuaderno', 'Lápiz', 'Carpeta'],
    'precio': [30.0, 5.0, 12.0]
}, index=['P10', 'P20', 'P40'])
precio_serie = productos['precio']
precio_tabla = productos[['precio']]
display(productos, precio_serie, precio_tabla)
print(type(precio_serie).__name__, precio_serie.ndim, precio_serie.shape)
print(type(precio_tabla).__name__, precio_tabla.ndim, precio_tabla.shape)
print('Un valor:', precio_serie.iloc[0])

Series 1 (3,)
DataFrame 2 (3, 1)
Un valor: 30.0


,producto,precio
P10,Cuaderno,30.0
P20,Lápiz,5.0
P40,Carpeta,12.0


P10    30.0
P20     5.0
P40    12.0
Name: precio, dtype: float64

,precio
P10,30.0
P20,5.0
P40,12.0


#### Nota Codex — Lectura del resultado

La Series tiene shape `(3,)` y ndim 1; la tabla tiene shape `(3, 1)` y ndim 2. Ambas contienen tres precios, pero conservan estructuras diferentes. `to_frame(name='precio')` permite convertir la Series en tabla. En Pandas II aplicarás la misma distinción al seleccionar filas.

## 2. Tamaño y metadatos: qué se conserva

#### Nota Codex — Concepto y uso

`len(df)` cuenta filas; `shape` contiene (filas, columnas); `size` cuenta celdas, incluso faltantes. Son atributos, salvo `len`, que es una función de Python. Esta es una lectura de la estructura; el diagnóstico del conjunto con info y describe se desarrolla en Pandas IV.

`to_numpy()` entrega valores sin index ni columns. No es una forma de exportar una tabla con todos sus metadatos.

In [3]:
print('Filas:', len(productos))
print('Forma:', productos.shape, 'Celdas:', productos.size)
print('Etiquetas:', productos.index.tolist())
print('Columnas:', productos.columns.tolist())
print('Nombre de la Series:', precio_serie.name)
valores = precio_serie.to_numpy()
print('Array sin etiquetas:', valores)
display(precio_serie.to_frame(name='precio_unitario'))

Filas: 3
Forma: (3, 2) Celdas: 6
Etiquetas: ['P10', 'P20', 'P40']
Columnas: ['producto', 'precio']
Nombre de la Series: precio
Array sin etiquetas: [30.  5. 12.]


,precio_unitario
P10,30.0
P20,5.0
P40,12.0


#### Nota Codex — Lectura del resultado

Hay 3 filas, 2 columnas y 6 celdas. El índice no es una tercera columna de datos. El array conserva 30, 5 y 12, pero ya no indica a qué producto corresponde cada valor. El nombre de columna se puede definir al construir una tabla desde una Series.

## 3. Alineación por etiquetas, no por orden

#### Nota Codex — Concepto y uso

Pandas usa las etiquetas para decidir qué valores pertenecen al mismo elemento. Esto es especialmente importante si dos Series llegan en órdenes distintos. La suma entre Series es una operación **por elemento alineado**: primero empareja etiquetas, luego calcula.

No confundas esto con una unión entre tablas: concat y merge se estudian en Pandas VI.

In [4]:
lunes = pd.Series([10, 20], index=['A', 'B'], name='lunes')
martes = pd.Series([3, 4], index=['B', 'A'], name='martes')
display(lunes, martes)
total = lunes + martes
display(total)
print('Sumar un escalar a cada valor:', (lunes + 1).to_dict())
print('Con arrays se suma por posición:', lunes.to_numpy() + martes.to_numpy())
assert total.loc['A'] == 14 and total.loc['B'] == 23

Sumar un escalar a cada valor: {'A': 11, 'B': 21}
Con arrays se suma por posición: [13 24]


A    10
B    20
Name: lunes, dtype: int64

B    3
A    4
Name: martes, dtype: int64

A    14
B    23
dtype: int64

#### Nota Codex — Lectura del resultado

A recibe 10 + 4 = 14 y B recibe 20 + 3 = 23. Sumar el escalar 1 actúa sobre cada valor (11 y 21), sin un bucle explícito: es una operación vectorizada. Los arrays dan 13 y 24 porque ya no tienen etiquetas. Quitar el índice cambia el significado de la operación; no es una corrección inocua.

## 4. Construir una tabla con Series incompletas

#### Nota Codex — Concepto y uso

Al pasar Series a un DataFrame, cada columna se alinea por índice. Una etiqueta que no existe en una Series deja una celda faltante en esa columna. En cambio, al pasar listas se emparejan posiciones y sus longitudes deben coincidir.

Un faltante significa que no tenemos ese dato; no afirma que el valor sea cero. No lo rellenaremos aquí: decidir su tratamiento corresponde a Pandas IV y V.

In [5]:
stock = pd.Series([8, 4], index=['A', 'B'], name='stock')
precio = pd.Series([12.0, 5.0], index=['B', 'C'], name='precio')
catalogo = pd.DataFrame({'stock': stock, 'precio': precio})
display(catalogo)
# Error intencional controlado: listas de diferente longitud.
try:
    pd.DataFrame({'stock': [8, 4], 'precio': [12.0]})
except ValueError as error:
    print('No se puede construir esa tabla:', error)

No se puede construir esa tabla: All arrays must be of the same length


,stock,precio
A,8.0,NaN
B,4.0,12.0
C,NaN,5.0


#### Nota Codex — Lectura del resultado

La tabla tiene las etiquetas A, B y C. Solo B tiene ambos datos: stock 4 y precio 12. A no tiene precio y C no tiene stock. El error de longitudes se captura para poder continuar Ejecutar todo; no se ignora en una carga real.

## 5. Tipos explícitos y valores faltantes

#### Nota Codex — Concepto y uso

`int64` y `Int64` no son el mismo dtype: Int64 (con I mayúscula) admite pd.NA. `boolean` y `string` también admiten pd.NA. Las fechas admiten NaT y category admite faltantes.

Aquí creamos objetos con el tipo correcto desde el principio; convertir o reparar columnas existentes se deja a Pandas V. Los códigos como "001" deben ser texto si los ceros forman parte del identificador.

In [6]:
unidades = pd.Series([2, pd.NA, 0], dtype='Int64', name='unidades')
confirmado = pd.Series([True, pd.NA, False], dtype='boolean')
codigos = pd.Series(['001', '010', pd.NA], dtype='string')
fechas = pd.Series([pd.Timestamp('2026-01-15'), pd.NaT], dtype='datetime64[ns]')
categorias = pd.Series(['local', None], dtype='category')
display(unidades, confirmado, codigos, fechas, categorias)
print('Cantidad desconocida:', pd.isna(unidades.iloc[1]))
print('Cantidad conocida igual a cero:', unidades.iloc[2])

Cantidad desconocida: True
Cantidad conocida igual a cero: 0


0       2
1    <NA>
2       0
Name: unidades, dtype: Int64

0     True
1     <NA>
2    False
dtype: boolean

0     001
1     010
2    <NA>
dtype: string

0   2026-01-15
1          NaT
dtype: datetime64[ns]

0    local
1      NaN
dtype: category
Categories (1, object): ['local']

#### Nota Codex — Lectura del resultado

Desconocido y cero son estados distintos. `pd.isna` permite reconocer el faltante; comparar con `== pd.NA` no ofrece una respuesta booleana ordinaria. No necesitas memorizar todas las representaciones: primero identifica el tipo y el significado del dato.

#### Nota Codex — Repaso y siguiente paso

Antes de continuar, explica por qué dos objetos con tres números pueden tener shape diferente y por qué cambiar el orden de una Series no cambia sus etiquetas. Sigue con Pandas II y luego con los ejercicios comunes al final de Pandas III.

#### Nota Codex — Límites del complemento

El diagnóstico de nulos, duplicados, frecuencias y estadísticos se estudia en **Pandas IV**.
La conversión y limpieza de columnas, imputación, recodificación y tratamiento de texto se estudian en **Pandas V**.
`concat` y `merge` se estudian en **Pandas VI**.
Los gráficos y su preparación están en [Visualización con Pandas](../../18_VISUALIZACION/02_PANDAS/01_Notebooks/04_Visualization_II_Pandas.ipynb). Aquí no se repiten esos temas.